# Data Scraping

Before any model can be built, the raw data needs to be collected. This section describes where the data comes from, what each source contains, and how the collection is run in practice.

The project uses three types of data: electricity prices, electricity consumption, and weather observations. All of it is fetched automatically from two public APIs using a single function call. The raw files are saved locally and used as input in the next step.

---
## Data sources

### 1. Day-ahead electricity prices
Fetched from **Energi Data Service** using two overlapping datasets — `Elspotprices` for older history and `DayAheadPrices` for more recent data. Both are normalised to the same column schema, merged, and deduplicated into a single file with one row per hour. Prices are available in both EUR/MWh and DKK/MWh.

DK1 covers western Denmark. Prices in this area are strongly influenced by wind power availability, interconnection flows to Germany and Norway, and hourly demand patterns — which makes them both relevant to forecast and challenging to predict.

### 2. Hourly electricity consumption
Also from **Energi Data Service**. The raw data comes at grid-area level and is aggregated to DK1 as a whole. Consumption is used as a proxy for grid stress: hours where system-level demand is high are treated as peak-load periods in the flexibility simulation later in the project.

### 3. Weather actuals
Fetched from the **Open-Meteo Archive API** for a representative DK1 location (lat 56.15, lon 8.45) using the ECMWF IFS model. Eight hourly variables are collected:

| Variable | Why it matters |
|----------|----------------|
| Wind speed and direction at 10 m and 100 m | Wind turbine production proxy — 100 m is closer to actual hub height |
| Shortwave radiation | Solar PV production proxy |
| Cloud cover | Affects how much solar radiation reaches panels |
| Temperature at 2 m | Drives heating and cooling demand |
| Mean sea level pressure | Captures large-scale weather regime shifts |

Weather actuals serve two purposes: they are used to estimate how large NWP forecast errors are at each lead time, and they form the base for generating synthetic forecasts for years where real forecast data is not available.

### 4. NWP weather forecasts
Fetched from the **Open-Meteo Previous Runs API**. This API stores the *as-issued* ECMWF forecast for each day, going back roughly 12 months. For each target hour it provides the forecast as it looked 1, 2, 3, 4, and 5 days in advance — stored as columns `_previous_day1` through `_previous_day5`.

These are real forecast values, not reanalysis, which makes them the best available source for measuring how large NWP errors are at each forecast horizon. The main limitation is that data is only available from January 2025 onwards. For the years before that, forecast values are simulated in the data processing step by adding horizon-scaled noise drawn from those real error distributions.

---
## Running the data collection

All four sources are collected in sequence by calling `fetch_all()`. It prints progress as it runs and saves each file to the `data/` folder. The weather forecast fetch is automatically limited to 2025 onwards since earlier data is not available from the API.

In [ ]:
from src.data.data_collection import fetch_all

results = fetch_all(start="2021-01-01", end="2026-04-28", price_area="DK1")

After the call completes, the `data/` folder contains four raw files:

| File | Content | Coverage |
|------|---------|----------|
| `weather_actuals_raw.csv` | Hourly weather observations for DK1 West | 2021 – present |
| `weather_forecasts_raw.csv` | NWP previous-run forecasts for DK1 West | 2025 – present |
| `consumption_dk1_raw.csv` | Hourly DK1 electricity consumption | 2021 – present |
| `day_ahead_prices_dk1_raw.csv` | Hourly DK1 day-ahead electricity prices | 2021 – present |

---
## Next step: Data processing

The raw files are not yet in a form the model can use. The next step, handled by `src/data/data_processing.py`, does two things:

1. **Estimate NWP error distributions** — compares the real forecasts against the actuals at five lead times (24, 48, 72, 96, and 120 hours) and saves summary statistics to `data/weather_error_distributions.csv`.

2. **Build the forecast dataset** — for every 12-hour issue time from 2021 to present, simulates a 120-hour weather forecast by adding horizon-scaled Gaussian noise to the actual weather values. The result is saved as `data/forecast_dataset.parquet` and becomes the direct input to the XGBoost model.